<a href="https://colab.research.google.com/github/namraaftab2006/neww/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install -q streamlit tensorflow scikit-learn pillow numpy

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DATASET_DIR = "/content/drive/MyDrive/images_compressed"

In [ ]:
FEATURE_FILE = "/content/drive/MyDrive/fashion_features.npz"

In [13]:
%%writefile /content/app.py
from pathlib import Path
import streamlit as st
import numpy as np
from PIL import Image
from sklearn.metrics.pairwise import cosine_similarity

from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

st.set_page_config(page_title="Fashion Finder", page_icon="👗", layout="wide")

st.title("👗 Fashion Finder")
st.write("Upload a clothing image to discover visually similar items.")

FEATURE_FILE = "/content/drive/MyDrive/fashion_features.npz"
DATASET_DIR = Path("/content/drive/MyDrive/images_compressed")


@st.cache_resource
def load_model():
    return MobileNetV2(weights="imagenet", include_top=False,
                       pooling="avg", input_shape=(224, 224, 3))


@st.cache_data
def load_database():
    data = np.load(FEATURE_FILE, allow_pickle=False)
    return data["features"], data["paths"].astype(str).tolist()


def extract_features(img, model):
    img = img.convert("RGB").resize((224, 224))
    arr = np.expand_dims(np.asarray(img, dtype=np.float32), axis=0)
    return model.predict(preprocess_input(arr), verbose=0)


if not Path(FEATURE_FILE).exists():
    st.error(f"Feature file not found: {FEATURE_FILE}")
    st.stop()

features, saved_paths = load_database()
model = load_model()

st.sidebar.header("Settings")
top_k = st.sidebar.slider("Number of recommendations", 1, 10, 5)

uploaded_file = st.file_uploader(
    "Choose or drag and drop a clothing image",
    type=["jpg", "jpeg", "png", "webp"]
)

if uploaded_file is not None:
    try:
        query_image = Image.open(uploaded_file).convert("RGB")
        st.subheader("Your uploaded image")
        st.image(query_image, width=300)

        if st.button("Find Similar Clothes", type="primary"):
            with st.spinner("Finding similar items..."):
                query_features = extract_features(query_image, model)
                scores = cosine_similarity(query_features, features)[0]

                # Rank everything first, then keep only the top results whose files exist
                order = np.argsort(scores)[::-1]
                ranked_indices = []
                for i in order:
                    if Path(saved_paths[i]).is_file():
                        ranked_indices.append(i)
                    if len(ranked_indices) == top_k:
                        break

            st.subheader("✨ Recommended clothing")

            if not ranked_indices:
                st.warning("No dataset images found. Check your saved image paths.")
            else:
                columns = st.columns(min(5, len(ranked_indices)))
                for position, idx in enumerate(ranked_indices):
                    with columns[position % len(columns)]:
                        try:
                            with Image.open(saved_paths[idx]) as result:
                                st.image(result.convert("RGB"), use_container_width=True)
                            st.caption(Path(saved_paths[idx]).name)
                            st.write(f"Similarity: {scores[idx]:.1%}")
                        except Exception:
                            st.caption("Could not display this image.")

    except Exception as e:
        st.error(f"Could not process the image: {e}")
else:
    st.info("Upload an image above to get started.")

Overwriting /content/app.py


In [14]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

In [15]:
import subprocess, time, re

# stop anything left from earlier runs
!pkill -f streamlit
!pkill -f cloudflared

st_log = open("/content/streamlit.log", "w")
subprocess.Popen(
    ["streamlit", "run", "/content/app.py",
     "--server.port", "8501",
     "--server.address", "0.0.0.0",
     "--server.headless", "true"],
    stdout=st_log, stderr=subprocess.STDOUT
)
time.sleep(8)

cf_log = open("/content/cloudflared.log", "w")
subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://localhost:8501"],
    stdout=cf_log, stderr=subprocess.STDOUT
)

url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", open("/content/cloudflared.log").read())
    if m:
        url = m.group(0)
        break

print("Open your app here:", url if url else "Tunnel failed, check /content/cloudflared.log")

Open your app here: https://lion-mug-welcome-kenneth.trycloudflare.com
